In [ ]:
# pandas leads to out-of-memory errors when datasets outgrow RAM
# use lazy evaluation to stream data with Polars -- 
#   transformations are queued, optimized via query planner, 
#   and executed only when explicitly called

In [48]:
!pip3 install polars


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [49]:
import polars as pl

In [ ]:
def lazy_transform(file_path, output_path):
    df = pl.scan_parquet(file_path)

    print(df.head())

    transformed_df = (
        df
        .with_columns(
            pl.col("Survived").cast(pl.String)
        )
        .with_columns(
            alive = pl.when(pl.col("Survived") == "1")
                                .then(pl.lit("Yes"))
                                .otherwise(pl.lit("No"))
                            )
        .drop(["SibSp", "Parch"])
        )

    transformed_df.sink_csv(output_path)


In [58]:
lazy_transform('sample_data/titanic.parquet', 'output/titanic.csv')

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SLICE[offset: 0, len: 5]
  Parquet SCAN [sample_data/titanic.parquet]
  PROJECT */12 COLUMNS
  ESTIMATED ROWS: 891
